# EX CNN - Fashion MNIST baseline (annotated)
**Purpose:** This notebook mirrors the original solution but adds short task descriptions so students can follow each step.

## 0) Setup: libraries and plotting
Import TensorFlow/Keras, Matplotlib, and standard helpers that every later cell relies on.

In [1]:
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, Flatten, Dense, Dropout, BatchNormalization, Activation
from keras.utils import to_categorical
from keras.activations import swish
import matplotlib.pyplot as plt
import os
import zipfile

I0000 00:00:1778080126.193667   96341 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778080126.194090   96341 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778080126.248791   96341 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778080128.673344   96341 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

## 1) Utility helpers: Drive mounting and packaging artifacts
Wrap Google Drive mounting for Colab runs and provide a helper that stores trained models and compresses them into a downloadable zip file.

In [2]:
def try_to_mount_drive():
    """Mount Google Drive. For local usage only."""
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        import os
        os.chdir('/content/drive/My Drive/Colab Notebooks')
        return True
    except ImportError:
        print("Google Colab module not found. Skipping Google Drive mount.")
        return False

def save_and_zip_models(trained_models, output_dir='models', zip_filename='models.zip'):
    """Save the trained models to disk and compress them into a zip file."""
    # Ensure the output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Save each trained model to the output directory with .keras extension
    for filename, model in trained_models.items():
        filepath = os.path.join(output_dir, f"{filename}.keras")
        model.save(filepath)
        print(f"Model saved to {filepath}")

    # Compress the model files into a zip file
    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                # Use arcname to avoid storing the full path in the zip file
                zipf.write(os.path.join(root, file), arcname=file)
    print(f"Models compressed into {zip_filename}")

## 2) Andmete ettevalmistus

Lae Fashion MNIST, vorminda pildid konvolutsioonivõrgule sobivaks ja loo kategoorilised sildid, et treening oleks täpselt samasugune kui lahenduses.

**Sinu ülesanne:**
- Kasuta `keras.datasets.fashion_mnist.load_data()` ja määra tulemused `(X_train, y_train), (X_test, y_test)` muutujatesse.
- Kuju muutmiseks kasuta `reshape((-1, 28, 28, 1))`, teisenda `astype("float32")` ning normaalseeri väärtused jagades 255-ga.
- Rakenda `to_categorical` mõlemale sildikomplektile ning kasuta `num_classes=10`.
- Tagasta `((X_train, y_train), (X_test, y_test))`, et ülejäänud toru saaks samu objekte taaskasutada.

**Kontrolli:**
- `X_train.shape == (60000, 28, 28, 1)` ja väärtuste vahemik on [0, 1].
- `y_train.shape == (60000, 10)` ning sama kehtib testandmete kohta.
- Funktsioon ei kasuta globaalseid muutujaid ja on puhas (s.t. alati tagastab uued massivviited).

In [3]:
def load_and_process_data():
    """Load the dataset and preprocess it."""
    (X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

    X_train = X_train.reshape((-1, 28, 28, 1)).astype("float32") / 255.0
    X_test = X_test.reshape((-1, 28, 28, 1)).astype("float32") / 255.0

    y_train = to_categorical(y_train, num_classes=10)
    y_test = to_categorical(y_test, num_classes=10)

    return (X_train, y_train), (X_test, y_test)


## 3) Mudeliarhitektuurid

Kirjuta kaks tehasfunktsiooni, mis loovad sama kihistruktuuri nagu originaalses lahenduses.

**Sinu ülesanne:**
- `create_basic_cnn` peab tagastama `Sequential` mudeli jadaga: `Conv2D(32, 3x3, relu, input_shape)` → `MaxPooling2D(2x2)` → `Conv2D(64, 3x3, relu)` → `MaxPooling2D(2x2)` → `Flatten` → `Dense(128, relu)` → `Dropout(0.5)` → `Dense(num_classes, softmax)`.
- `create_advanced_cnn` kordab lahenduse varianti: `Conv2D(32, padding='same')` → `BatchNormalization` → `Activation(swish)` → `Conv2D(64, padding='same')` → `BatchNormalization` → `Activation(swish)` → `MaxPooling2D(2x2)` → `Dropout(0.3)` → `Flatten` → `Dense(128)` → `BatchNormalization` → `Activation(swish)` → `Dropout(0.5)` → `Dense(num_classes, softmax)`.
- Mõlemas funktsioonis kasuta edasiantud `input_shape` ja `num_classes` parameetreid ning ära kompileeri mudeleid.

**Kontrolli:**
- `model.output_shape[-1] == num_classes` ja viimane kiht on softmax.
- Dropouti ja normaliseerimise määrad ühtivad lahenduses tooduga.
- Funktsioonid tagastavad värskelt loodud mudeleid (neid saab mitu korda kutsuda).

In [4]:
def create_basic_cnn(input_shape, num_classes):
    """Create a basic CNN model."""
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])


def create_advanced_cnn(input_shape, num_classes):
    """Create a more advanced CNN model with batch normalization."""
    return Sequential([
        Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        BatchNormalization(),
        Activation(swish),
        Conv2D(64, (3, 3), padding='same'),
        BatchNormalization(),
        Activation(swish),
        MaxPooling2D((2, 2)),
        Dropout(0.3),
        Flatten(),
        Dense(128),
        BatchNormalization(),
        Activation(swish),
        Dropout(0.5),
        Dense(num_classes, activation='softmax'),
    ])


## 4) Treeningu abifunktsioonid

Kasuta samu visualiseerimis- ja treenimisplokke, et tekitada mudelite võrdlusgraafikud ning salvestada treeningu ajalugu.

**Sinu ülesanne:**
- `plot_training_histories(histories)` loob `plt.figure(figsize=(15, 5))`, seab kaks alamgraafikut ja joonistab igale mudelile nii treeningu kui ka valideerimise täpsuse ning kaotuse.
- `train_models(models, train_data, test_data, epochs=20, batch_size=64)` võtab mudelite sõnastiku, kompileerib iga mudeli `optimizer='adam'`, `loss='categorical_crossentropy'`, `metrics=['accuracy']`, treenib neid `model.fit` abil ning salvestab nii treenitud mudeli kui ka `history.history` sisu eraldi sõnastikesse.
- Funktsioon prindib `model.evaluate` tulemuse igale mudelile ja tagastab `(trained_models, histories)`.

**Kontrolli:**
- `histories[model_name]` sisaldab võtmeid `accuracy`, `loss`, `val_accuracy`, `val_loss`.
- Mudelite nimed väljatrükkides ühtivad sisend-sõnastikuga ja neid saab hiljem salvestada.
- Graafikud renderduvad kahes veerus ning legendid eristavad treeningut ja valideerimist.

In [5]:
def plot_training_histories(histories):
    """Plot training histories for all models."""
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    for model_name, history in histories.items():
        plt.plot(history['accuracy'], label=f'{model_name} train')
        plt.plot(history['val_accuracy'], linestyle='--', label=f'{model_name} val')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    for model_name, history in histories.items():
        plt.plot(history['loss'], label=f'{model_name} train')
        plt.plot(history['val_loss'], linestyle='--', label=f'{model_name} val')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()


def train_models(models, train_data, test_data, epochs=20, batch_size=64):
    """Train multiple models and return them along with their histories."""
    X_train, y_train = train_data
    X_test, y_test = test_data

    trained_models = {}
    histories = {}

    for model_name, model in models.items():
        print(f'\nTraining model: {model_name}')

        model.compile(
            optimizer='adam',
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        history = model.fit(
            X_train,
            y_train,
            validation_data=(X_test, y_test),
            epochs=epochs,
            batch_size=batch_size,
            verbose=1,
        )

        loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
        print(f'{model_name} -> test loss: {loss:.4f}, test accuracy: {accuracy:.4f}')

        trained_models[model_name] = model
        histories[model_name] = history.history

    return trained_models, histories


## 5) Peatoru: treeni ja paki mudelid

Lõpuosas kopeeri täpselt sama juhtloogika: Drive'i ühendamine, treening ainult lokaalses režiimis ning mudelite arhiveerimine.

**Sinu ülesanne:**
- Kutsu `try_to_mount_drive()` ja salvesta tulemus `local` muutujasse; kõik järgnevad sammud toimuvad `if local:` haru sees.
- Laadi ja töötle andmed, arvuta `input_shape` ning `num_classes`, loo `models = {'basic_cnn': ..., 'advanced_cnn': ...}` ning treeni need `train_models` funktsiooniga.
- Kuva graafikud `plot_training_histories(histories)` abil, lisa oma analüüsikommentaarid ning lõpuks kutsu `save_and_zip_models(trained_models)`, et `.keras` failid saaksid zip'i pakitud.

**Kontrolli:**
- Treening ei käivitu, kui Drive'i mount ebaõnnestub.
- Kõik sammud (laadimine, mudeli loomine, treenimine, graafikud, salvestamine) on selges järjekorras ühe koodiploki sees.

In [6]:
local = try_to_mount_drive()

# Make sure all the training logic has the correct indentation, otherwise tester will try to run the logic and error.
if local:
    (X_train, y_train), (X_test, y_test) = load_and_process_data()
    input_shape = X_train.shape[1:]
    num_classes = y_train.shape[1]

    models = {
        'basic_cnn': create_basic_cnn(input_shape, num_classes),
        'advanced_cnn': create_advanced_cnn(input_shape, num_classes),
    }

    trained_models, histories = train_models(
        models,
        train_data=(X_train, y_train),
        test_data=(X_test, y_test),
        epochs=10,
        batch_size=64,
    )

    plot_training_histories(histories)

    print('\nQuick comparison by validation accuracy:')
    for model_name, history in histories.items():
        best_val_acc = max(history.get('val_accuracy', [0]))
        print(f'{model_name}: best val_accuracy = {best_val_acc:.4f}')

    save_and_zip_models(trained_models)


Google Colab module not found. Skipping Google Drive mount.


## 6) Mudelite jagamine testimiseks

Kuna skript ühendab Google Drive'i ja salvestab zip-faili kataloogi `/content/drive/My Drive/Colab Notebooks`, asub fail `models.zip` pärast käivitamist automaatselt teie Drive'is. Testijale juurdepääsu andmiseks:

1. Paremklõpsake failil `models.zip` Drive'is.
2. Klõpsake **Share**.
3. Jaotises **General access** valige **„Anyone with the link“**.
4. Kopeerige jagatud lingi URL. Näide:  
   `https://drive.google.com/file/d/1ABC123DEFghIJK/view`  
   Siin on faili ID `1ABC123DEFghIJK`.
5. Asendage muutuja `models_download_id` väärtus selle ID-ga (real `models_download_id = "1-ASENDA_Oma_DRIVE_FAILI_ID_FIXME"`).

Seejärel saab testija teie ID-d kasutades mudelid alla laadida ja teie tööd hinnata.

In [7]:
models_download_id = "1-REPLACE_WITH_YOUR_DRIVE_FILE_ID_FIXME"